# 💻 Laptop Price Predictor
Predicting laptop prices (in Euros) using brand, CPU, RAM, storage, and OS.

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
import pickle

In [2]:
df = pd.read_csv('laptop_price.csv', encoding='latin1', sep=',')
df.head()

,Company,Product,TypeName,Inches,ScreenResolution,CPU_Company,CPU_Type,CPU_Frequency (GHz),RAM (GB),Memory,GPU_Company,GPU_Type,OpSys,Weight (kg),Price (Euro)
0,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel,Core i5,2.3,8,128GB SSD,Intel,Iris Plus Graphics 640,macOS,1.37,1339.69
1,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel,Core i5,1.8,8,128GB Flash Storage,Intel,HD Graphics 6000,macOS,1.34,898.94
2,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel,Core i5 7200U,2.5,8,256GB SSD,Intel,HD Graphics 620,No OS,1.86,575.00
3,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel,Core i7,2.7,16,512GB SSD,AMD,Radeon Pro 455,macOS,1.83,2537.45
4,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel,Core i5,3.1,8,256GB SSD,Intel,Iris Plus Graphics 650,macOS,1.37,1803.60


## Feature Engineering
The `Memory` column has messy values like `'128GB SSD + 1TB HDD'`.
We extract:
- `storage_gb` → total storage in GB
- `storage_type` → SSD / HDD / SSD+HDD / Hybrid / Flash

In [3]:
def parse_storage_gb(mem):
    total = 0
    for part in str(mem).split('+'):
        part = part.strip()
        if 'TB' in part:
            try: total += float(part.split('TB')[0].strip()) * 1000
            except: pass
        elif 'GB' in part:
            try: total += float(part.split('GB')[0].strip())
            except: pass
    return int(total)

def parse_storage_type(mem):
    m = str(mem).upper()
    if 'SSD' in m and 'HDD' in m: return 'SSD+HDD'
    elif 'SSD' in m:              return 'SSD'
    elif 'HDD' in m:              return 'HDD'
    elif 'HYBRID' in m:           return 'Hybrid'
    elif 'FLASH' in m:            return 'Flash'
    else:                         return 'Other'

df['storage_gb']   = df['Memory'].apply(parse_storage_gb)
df['storage_type'] = df['Memory'].apply(parse_storage_type)

print(df['storage_gb'].value_counts().head(8))
print('---')
print(df['storage_type'].value_counts())

storage_gb
256     420
1000    237
500     124
512     118
1128     94
128      79
1256     74
32       43
Name: count, dtype: int64
---
storage_type
SSD        637
HDD        360
SSD+HDD    200
Flash       70
Hybrid       8
Name: count, dtype: int64


In [4]:
# Simplify CPU: 'Core i5 7200U' -> 'Core i5'
def simplify_cpu(cpu):
    for label in ['Core i3','Core i5','Core i7','Core i9',
                  'Ryzen 3','Ryzen 5','Ryzen 7','Ryzen 9',
                  'Core M','Celeron','Pentium','Atom','A-Series','E-Series']:
        if label in str(cpu):
            return label
    return 'Other'

df['cpu_simple'] = df['CPU_Type'].apply(simplify_cpu)
print(df['cpu_simple'].value_counts())

cpu_simple
Core i7     515
Core i5     423
Core i3     134
Celeron      78
Other        56
Pentium      30
Core M       17
Atom         13
E-Series      9
Name: count, dtype: int64


In [5]:
# Simplify OS
def simplify_os(os):
    os = str(os)
    if 'Windows' in os:              return 'Windows'
    elif 'macOS' in os or 'Mac' in os: return 'macOS'
    elif 'Linux' in os:              return 'Linux'
    elif 'Chrome' in os:             return 'Chrome OS'
    else:                            return 'Other'

df['os_simple'] = df['OpSys'].apply(simplify_os)
print(df['os_simple'].value_counts())

os_simple
Windows      1101
Other          68
Linux          58
Chrome OS      27
macOS          21
Name: count, dtype: int64


In [6]:
# Encoding dictionaries
d1 = {
    'Dell':0,'Lenovo':1,'HP':2,'Asus':3,'Acer':4,
    'MSI':5,'Toshiba':6,'Apple':7,'Samsung':8,'Microsoft':9,
    'Razer':10,'Xiaomi':11,'Huawei':12,'Google':13,'LG':14,
    'Fujitsu':15,'Mediacom':16,'Vero':17,'Chuwi':18
}
d2 = {
    'Core i3':0,'Core i5':1,'Core i7':2,'Core i9':3,
    'Ryzen 3':4,'Ryzen 5':5,'Ryzen 7':6,'Ryzen 9':7,
    'Core M':8,'Celeron':9,'Pentium':10,'Atom':11,
    'A-Series':12,'E-Series':13,'Other':14
}
d3 = {'HDD':0,'SSD':1,'SSD+HDD':2,'Hybrid':3,'Flash':4,'Other':5}
d4 = {'Windows':0,'macOS':1,'Linux':2,'Chrome OS':3,'Other':4}

df['company_enc']      = df['Company'].map(d1)
df['cpu_enc']          = df['cpu_simple'].map(d2)
df['storage_type_enc'] = df['storage_type'].map(d3)
df['os_enc']           = df['os_simple'].map(d4)

print('Encoding done!')

Encoding done!


In [7]:
# Prepare features and target
X = df[['company_enc','cpu_enc','RAM (GB)','storage_gb','storage_type_enc','os_enc']]
y = df['Price (Euro)']

mask = X.notna().all(axis=1)
X = X[mask]
y = y[mask]

print('Training samples:', len(X))
print(X.head())

Training samples: 1275
   company_enc  cpu_enc  RAM (GB)  storage_gb  storage_type_enc  os_enc
0            7        1         8         128                 1       1
1            7        1         8         128                 4       1
2            2        1         8         256                 1       4
3            7        2        16         512                 1       1
4            7        1         8         256                 1       1


In [8]:
# Train and compare models
models = {
    'LinReg' : LinearRegression(),
    'KNN-3'  : KNeighborsRegressor(n_neighbors=3),
    'KNN-5'  : KNeighborsRegressor(n_neighbors=5),
    'SVM-RBF': SVR(kernel='rbf')
}

for name, model in models.items():
    model.fit(X, y)
    score = model.score(X, y)
    print(f'{name}: {score:.4f}')

LinReg: 0.5960
KNN-3: 0.6579
KNN-5: 0.6611
SVM-RBF: -0.0410


In [9]:
# Pick the best model (check scores above — usually KNN-3)
final_model = models['KNN-3']

In [10]:
# Quick test — Dell, Core i5, 8GB RAM, 256GB SSD, Windows
test = [[d1['Dell'], d2['Core i5'], 8, 256, d3['SSD'], d4['Windows']]]
yp = int(final_model.predict(test)[0])
print(f'Predicted Price: €{yp}')

Predicted Price: €782


C:\Users\skull\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(


In [11]:
pickle.dump(final_model, open('laptop_model.pkl', 'wb'))
print('Model saved as laptop_model.pkl')

Model saved as laptop_model.pkl
